# Somo la 11 - Itifaki ya Muktadha wa Mfano (MCP)

**Itifaki ya Muktadha wa Mfano (MCP)** ni kiwango wazi kinachowezesha maajenti kugundua na kutumia zana, rasilimali, na vyanzo vya data kwa wakati halisi wa utekelezaji. Badala ya kuweka zana moja kwa moja ndani ya ajenti, MCP inawawezesha maajenti kuunganishwa na seva za nje zinazotoa uwezo kwa mahitaji.

Katika somo hili, utajifunza:
- MCP ni nini na kwa nini ni muhimu kwa mifumo ya maajenti
- Jinsi usanifu wa mteja-seva wa MCP unavyofanya kazi
- Jinsi ya kujenga maajenti yanayotumia ugunduzi wa zana kwa mtindo wa MCP


## Usanidi

**Masharti ya awali:**
- Mradi wa Microsoft Foundry wenye mfano uliowekwa
- Endesha `az login` kwa ajili ya uthibitishaji


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ni Nini Itifaki ya Muktadha wa Mfano (MCP)?

MCP inaelezea njia ya kawaida kwa maajenti wa AI kugundua na kuwasiliana na zana za nje na vyanzo vya data:

- **MCP Server**: Inaonyesha zana, rasilimali, na misukumo kupitia itifaki ya kawaida
- **MCP Client**: Muda wa kuendesha maajenti unaounganisha kwenye seva na kugundua uwezo uliopo
- **Dynamic Discovery**: Maajenti hayawezi kuhitaji zana zilizoandikwa kali — hugundua kinachopatikana wakati wa kuendesha

Hii ni yenye nguvu kwa kujenga mifumo ya maajenti inayoweza kupanuka ambapo uwezo mpya unaweza kuongezwa bila kubadilisha msimbo wa maajenti.


## Jinsi MCP Inavyofanya Kazi

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Wakala (mteja MCP) anahusiana na seva ya MCP
2. Seva huwajibika na orodha ya zana zilizopatikana na miundo yao
3. Wakala anaweza kisha kuita zana yoyote aliyogundua wakati wa kufikiri kwake
4. Matokeo huirudi kupitia itifaki ile ile


## Kuiga Ugunduzi wa Zana za MCP

Kwa kuwa seva halisi ya MCP inahitaji mchakato wa seva unaoendesha, tutajifunza mfano huu tukitumia kazi za `@tool` zinazokirirudia kile huduma ya makazi inayounganishwa na MCP ingetoa.

Katika uzalishaji, zana hizi zingegunduliwa kwa nguvu kutoka kwa seva ya MCP badala ya kufafanuliwa kiasili.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Kujenga Wakala kwa Vyombo vya Mtindo wa MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP katika Uzalishaji

Katika mazingira ya uzalishaji, MCP inawezesha mifumo yenye nguvu:

- **Ugunduzi wa zana kwa nguvu za mabadiliko**: Wakala huunganisha na seva za MCP na kugundua zana wakati wa kukimbia
- **Miundo iliyotenganishwa**: Watoa huduma za zana wanaweza kusasisha kwa uhuru bila kuathiri wakala
- **Kushirikiana kwa mashirika mbalimbali**: Timu zinaweza kufichua uwezo kupitia seva za MCP ambazo wakala yeyote anaweza kutumia
- **Msaada wa Mfumo wa Wakala wa Microsoft**: MAF ina msaada wa mteja wa MCP uliojengwa kupitia muunganisho wa `mcp`

Kutumia seva halisi ya MCP na MAF, utakuwa unajiunga kupitia `hosted_mcp_tool()` au muunganisho wa mteja MCP.

**Jifunze zaidi:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Muhtasari

Katika somo hili, ulijifunza:
- **MCP** ni kiwango wazi cha kugundua zana kwa nguvu kati ya maajenti na watoaji wa zana
- **Muundo wa mteja-server** huruhusu maajenti kugundua uwezo wakati wa utekelezaji
- MCP huruhusu **mifumo ya maajenti inayoweza kuongezwa na isiyoshikamana** ambapo zana zinaweza kuongezwa bila mabadiliko ya msimbo
- Microsoft Agent Framework hutoa **msaada wa MCP uliojengewa ndani** kwa matumizi ya uzalishaji


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
